# Responses API

## Chat Completions에서 Responses API로 넘어가기

03번 노트북에서는 Chat Completions API로 기본 질문 응답, 고객 피드백 분석, PII 추출을 실습했습니다. 이 노트북에서는 같은 종류의 작업을 Responses API 방식으로 다시 구현합니다.

Responses API는 단순한 텍스트 생성뿐 아니라 이전 응답 이어가기, 출력 형식 제어, 향후 도구 확장까지 하나의 응답 중심 인터페이스로 다루기 위한 API입니다. 이번 워크숍에서는 APIM gateway를 통해 Azure OpenAI Responses API를 호출합니다.

## 목차

1. Responses API란?
2. Chat Completions와 차이
3. 기본 질문 응답
4. 감정 분석
5. PII 추출
6. 출력 형식 고정
7. 이전 대화 이어가기

## 1. Responses API란?

Responses API는 모델에게 입력을 보내고 응답을 받는 통합 API입니다. Chat Completions처럼 대화형 응답을 만들 수 있지만, 응답 객체를 중심으로 상태를 이어가거나 출력 형식을 더 명확히 관리하는 흐름에 맞춰 설계되어 있습니다.

이 노트북에서는 다음 관점으로 Responses API를 봅니다.

- Chat Completions에서 하던 작업을 Responses API로 어떻게 옮기는가
- `messages` 리스트 대신 `instructions`와 `input`에 무엇을 넣는가
- 응답 텍스트를 어떻게 꺼내고 후속 요청에 어떻게 연결하는가
- JSON 같은 고정 출력 형식을 어떻게 요청하고 검증하는가

## 2. Chat Completions와 차이

03번의 Chat Completions API는 보통 다음처럼 호출했습니다.

```python
client.chat.completions.create(
    model=CHAT_COMPLETIONS_MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt},
    ],
)
```

Python 코드에서 `messages`는 `list[dict]` 형태입니다. 각 원소는 `role`과 `content`를 가진 딕셔너리이고, API 요청으로 전송될 때는 JSON array가 됩니다.

Responses API에서는 지시문과 사용자 입력을 더 직접적으로 나눌 수 있습니다.

```python
client.responses.create(
    model=response_model,
    instructions="You are a helpful assistant.",
    input=prompt,
)
```

| 구분 | Chat Completions | Responses API |
|---|---|---|
| 기본 입력 | `messages` 리스트 | `instructions` 지시문 + `input` 입력 |
| 역할 분리 | `system`/`user`/`assistant` 메시지 | `instructions`와 `input` 분리 |
| 응답 접근 | `response.choices[0].message.content` | `response.output_text` 또는 `response.output` |
| 이어 말하기 | 이전 메시지 목록을 직접 관리 | `previous_response_id`로 이전 응답 연결 가능 |
| 확장 방향 | 채팅 메시지 중심 | 응답, 상태, 도구 확장을 포괄하는 인터페이스 |

이번 워크숍 APIM 환경에서는 안정적으로 사용할 수 있는 `responses.create()` 중심으로 실습합니다.

## APIM 환경 설정

`.env`에서 APIM gateway endpoint, APIM 구독 키, API version, 배포 이름을 읽습니다. APIM을 통과해야 하므로 `Ocp-Apim-Subscription-Key` 헤더를 함께 넣습니다.

In [25]:
from openai import AzureOpenAI
import json
import os
import re
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import Markdown, display

# 환경 변수 로드
dotenv_path = Path.cwd() / ".env"
load_dotenv(dotenv_path=dotenv_path, override=True)

azure_openai_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
azure_openai_key = os.getenv("AZURE_OPENAI_KEY")
azure_openai_api_version = os.getenv("AZURE_OPENAI_API_VERSION", "2025-04-01-preview")
response_model = os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME", "gpt-5.4-mini")

if not azure_openai_endpoint or not azure_openai_key:
    raise ValueError(".env 파일에 AZURE_OPENAI_ENDPOINT와 AZURE_OPENAI_KEY를 설정하세요.")

client = AzureOpenAI(
    azure_endpoint=azure_openai_endpoint,
    api_key=azure_openai_key,
    api_version=azure_openai_api_version,
    default_headers={"Ocp-Apim-Subscription-Key": azure_openai_key},
)

def get_response_text(response) -> str:
    """Responses API 응답에서 텍스트를 안전하게 꺼냅니다."""
    output_text = getattr(response, "output_text", None)
    if output_text:
        return output_text

    text_parts = []
    for output in getattr(response, "output", []):
        for content in getattr(output, "content", []):
            text = getattr(content, "text", None)
            if text:
                text_parts.append(text)

    return "\n".join(text_parts)

def display_response(response):
    """Responses API 응답을 Markdown으로 표시합니다."""
    text = get_response_text(response)
    display(Markdown(text))
    return text

def extract_json_object(text: str) -> dict:
    """모델 응답에서 JSON 객체를 추출해 파싱합니다."""
    cleaned = text.strip()
    cleaned = re.sub(r"^```(?:json)?\s*", "", cleaned)
    cleaned = re.sub(r"\s*```$", "", cleaned)

    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", cleaned, flags=re.DOTALL)
        if not match:
            raise
        return json.loads(match.group(0))

print(f"Azure OpenAI endpoint: {azure_openai_endpoint}")
print(f"API version: {azure_openai_api_version}")
print(f"Responses deployment: {response_model}")

Azure OpenAI endpoint: https://apim-ai-workshop-010.azure-api.net/
API version: 2025-04-01-preview
Responses deployment: gpt-5.4-mini


## 3. 기본 질문 응답

가장 단순한 Responses API 호출입니다. Chat Completions에서는 `messages` 리스트에 역할별 메시지를 넣었지만, Responses API에서는 `instructions`에 모델 지시문을 넣고 `input`에 실제 사용자 입력을 넣어 시작할 수 있습니다.

In [26]:
basic_instructions = """당신은 Azure OpenAI를 처음 배우는 수강생을 돕는 친절한 강사입니다.
기술 용어를 쓰더라도 한 문장 안에서 쉽게 풀어 설명하세요."""

basic_input = "Responses API를 처음 배우는 사람에게 한 문단으로 설명해줘."

basic_response = client.responses.create(
    model=response_model,
    instructions=basic_instructions,
    input=basic_input,
)

display_response(basic_response)

Responses API는 Azure OpenAI에서 **모델과 대화하거나 작업을 시키는 데 쓰는 새로운 통합형 API**라고 생각하면 됩니다. 쉽게 말해, 예전에는 텍스트 생성, 채팅, 도구 호출 같은 기능을 각각 따로 다뤄야 했다면, Responses API는 이런 기능들을 한곳에 모아 **질문을 보내고, 모델의 답을 받고, 필요하면 함수 호출이나 파일/이미지 같은 추가 작업까지 이어서 처리**할 수 있게 해줍니다. 그래서 개발자는 “메시지 주고받기”를 더 일관된 방식으로 구현할 수 있고, 사용자는 더 자연스럽게 AI와 상호작용할 수 있습니다. 특히 Azure OpenAI를 처음 시작하는 사람에게는, Responses API를 **AI에게 작업을 요청하는 표준 창구**라고 이해하면 가장 쉽습니다.

'Responses API는 Azure OpenAI에서 **모델과 대화하거나 작업을 시키는 데 쓰는 새로운 통합형 API**라고 생각하면 됩니다. 쉽게 말해, 예전에는 텍스트 생성, 채팅, 도구 호출 같은 기능을 각각 따로 다뤄야 했다면, Responses API는 이런 기능들을 한곳에 모아 **질문을 보내고, 모델의 답을 받고, 필요하면 함수 호출이나 파일/이미지 같은 추가 작업까지 이어서 처리**할 수 있게 해줍니다. 그래서 개발자는 “메시지 주고받기”를 더 일관된 방식으로 구현할 수 있고, 사용자는 더 자연스럽게 AI와 상호작용할 수 있습니다. 특히 Azure OpenAI를 처음 시작하는 사람에게는, Responses API를 **AI에게 작업을 요청하는 표준 창구**라고 이해하면 가장 쉽습니다.'

## 4. 감정 분석

03번 Chat Completions 노트북의 고객 피드백 분석 흐름을 Responses API로 다시 작성합니다. 같은 작업을 하되 호출 방식만 `client.responses.create()`로 바꿉니다.

In [27]:
review = """배송은 빨랐지만 제품 포장이 찢어져 있었고, 고객센터 답변도 늦었습니다. 다시 구매할지는 고민됩니다."""

sentiment_instructions = """당신은 고객 피드백을 분석하는 고객 지원 전문가입니다.
입력된 고객 리뷰의 감정을 분석하세요.

다음 항목을 한국어로 답하세요.
- 전체 감정: 긍정/부정/중립 중 하나
- 근거: 감정을 판단한 이유
- 후속 조치: 고객 지원팀이 하면 좋을 행동"""

sentiment_response = client.responses.create(
    model=response_model,
    instructions=sentiment_instructions,
    input=review,
)

display_response(sentiment_response)

- 전체 감정: **부정**
- 근거: 배송 속도는 좋았지만, **제품 포장 파손**과 **고객센터 응답 지연**이 불만으로 제기되어 있으며, 재구매 의사도 망설이고 있어 전반적으로 부정적인 감정이 우세합니다.
- 후속 조치: **포장 상태와 배송 과정**을 점검하고, 고객에게 **사과 및 보상/교환 가능 여부**를 안내하세요. 또한 고객센터 **응답 속도 개선**이 필요하며, 해당 고객에게는 **재구매 유도보다 먼저 불편 해소**에 집중하는 것이 좋습니다.

'- 전체 감정: **부정**\n- 근거: 배송 속도는 좋았지만, **제품 포장 파손**과 **고객센터 응답 지연**이 불만으로 제기되어 있으며, 재구매 의사도 망설이고 있어 전반적으로 부정적인 감정이 우세합니다.\n- 후속 조치: **포장 상태와 배송 과정**을 점검하고, 고객에게 **사과 및 보상/교환 가능 여부**를 안내하세요. 또한 고객센터 **응답 속도 개선**이 필요하며, 해당 고객에게는 **재구매 유도보다 먼저 불편 해소**에 집중하는 것이 좋습니다.'

## 5. PII 추출

PII(Personally Identifiable Information)는 이름, 이메일, 전화번호, 주소처럼 개인을 식별할 수 있는 정보입니다. Responses API에서도 Chat Completions와 마찬가지로 텍스트에서 PII 후보를 추출하도록 요청할 수 있습니다.

In [28]:
message_text = """
안녕하세요. 저는 김민수이고, 연락처는 010-1234-5678입니다.
계정 이메일은 minsu.kim@example.com이고, 배송지는 서울시 중구 세종대로 110입니다.
주문 번호 A-39482 관련해서 환불 상태를 확인하고 싶습니다.
"""

pii_instructions = """
당신은 개인정보(PII) 탐지 전문가입니다.
입력 문장에서 PII 후보를 추출하세요.

다음 형식으로 한국어로 답하세요.
- 이름
- 전화번호
- 이메일
- 주소
- 기타 식별 가능 정보

해당 항목이 없으면 "없음"이라고 답하세요.
"""

pii_response = client.responses.create(
    model=response_model,
    instructions=pii_instructions,
    input=message_text,
)

display_response(pii_response)

- 이름: 김민수
- 전화번호: 010-1234-5678
- 이메일: minsu.kim@example.com
- 주소: 서울시 중구 세종대로 110
- 기타 식별 가능 정보: 주문 번호 A-39482

'- 이름: 김민수\n- 전화번호: 010-1234-5678\n- 이메일: minsu.kim@example.com\n- 주소: 서울시 중구 세종대로 110\n- 기타 식별 가능 정보: 주문 번호 A-39482'

## 6. 출력 형식 고정

실무에서는 모델 응답을 사람이 읽는 문장보다 프로그램이 처리할 수 있는 JSON 형태로 받아야 할 때가 많습니다. 여기서는 모델에게 JSON 객체만 출력하도록 지시하고, Python에서 실제로 파싱해 봅니다.

모델 출력은 항상 검증해야 합니다. 아래 셀은 응답에서 JSON 객체를 추출해 `json.loads()`로 파싱합니다.

In [29]:
structured_instructions = """
입력된 고객 리뷰를 분석하고 JSON 객체 하나만 출력하세요.
마크다운 코드블록은 사용하지 마세요.

JSON 스키마:
{
  "sentiment": "positive | neutral | negative",
  "confidence": 0.0,
  "summary": "한 문장 요약",
  "recommended_action": "권장 조치"
}
"""

structured_response = client.responses.create(
    model=response_model,
    instructions=structured_instructions,
    input=review,
)

structured_text = get_response_text(structured_response)
print(structured_text)

structured_result = extract_json_object(structured_text)
structured_result

{
  "sentiment": "negative",
  "confidence": 0.96,
  "summary": "배송은 빨랐지만 제품 포장 손상과 고객센터 응답 지연으로 전반적인 만족도가 낮습니다.",
  "recommended_action": "포장 상태 점검 및 고객센터 응답 속도 개선이 필요하며, 해당 고객에게는 사과와 보상 또는 교환/환불 안내를 제공하세요."
}


{'sentiment': 'negative',
 'confidence': 0.96,
 'summary': '배송은 빨랐지만 제품 포장 손상과 고객센터 응답 지연으로 전반적인 만족도가 낮습니다.',
 'recommended_action': '포장 상태 점검 및 고객센터 응답 속도 개선이 필요하며, 해당 고객에게는 사과와 보상 또는 교환/환불 안내를 제공하세요.'}

## 7. 이전 대화 이어가기

Responses API는 이전 응답의 ID를 사용해 대화를 이어갈 수 있습니다. Chat Completions에서는 이전 메시지 목록을 애플리케이션이 직접 관리하는 경우가 많지만, Responses API에서는 `previous_response_id`로 앞선 응답을 참조할 수 있습니다.

현재 APIM 워크숍 환경에서는 `POST /responses` 중심 실습을 사용합니다. `GET /responses/{id}` 조회는 APIM operation이 별도로 열려 있지 않으면 404가 날 수 있으므로 여기서는 사용하지 않습니다.

In [30]:
conversation_instructions = """당신은 API 개념을 간결하게 설명하는 기술 강사입니다.
핵심만 짧고 명확하게 답하세요."""

conversation_input = "Responses API의 장점 두 가지를 짧게 설명해줘."

conversation_response = client.responses.create(
    model=response_model,
    instructions=conversation_instructions,
    input=conversation_input,
)

display_response(conversation_response)

Responses API의 장점 두 가지는:

1. **한 번의 API로 다양한 작업 지원**  
   텍스트 생성, 도구 호출, 멀티모달 처리를 더 통합적으로 다룰 수 있습니다.

2. **대화 상태와 흐름 관리가 쉬움**  
   이전 응답을 이어서 활용하기 편해, 복잡한 에이전트형 워크플로를 만들기 좋습니다.

'Responses API의 장점 두 가지는:\n\n1. **한 번의 API로 다양한 작업 지원**  \n   텍스트 생성, 도구 호출, 멀티모달 처리를 더 통합적으로 다룰 수 있습니다.\n\n2. **대화 상태와 흐름 관리가 쉬움**  \n   이전 응답을 이어서 활용하기 편해, 복잡한 에이전트형 워크플로를 만들기 좋습니다.'

In [31]:
followup_instructions = """이전 응답의 맥락을 유지하면서 질문에 답하세요.
Chat Completions와 Responses API의 차이를 수강생이 이해하기 쉽게 설명하세요."""

followup_input = "방금 답변을 Chat Completions API와 비교해서 한 문단으로 다시 정리해줘."

followup_response = client.responses.create(
    model=response_model,
    instructions=followup_instructions,
    input=followup_input,
    previous_response_id=conversation_response.id,
)

display_response(followup_response)

Chat Completions API는 주로 **대화형 텍스트 생성**에 초점이 맞춰져 있는 반면, **Responses API**는 텍스트 생성뿐 아니라 **도구 호출, 멀티모달 처리, 더 유연한 응답 흐름 관리**까지 한 번에 다루기 좋습니다. 그래서 단순한 채팅 기능만 필요하면 Chat Completions API도 충분하지만, 여러 작업을 연결하는 **복잡한 애플리케이션이나 에이전트형 기능**을 만들 때는 Responses API가 더 편리합니다.

'Chat Completions API는 주로 **대화형 텍스트 생성**에 초점이 맞춰져 있는 반면, **Responses API**는 텍스트 생성뿐 아니라 **도구 호출, 멀티모달 처리, 더 유연한 응답 흐름 관리**까지 한 번에 다루기 좋습니다. 그래서 단순한 채팅 기능만 필요하면 Chat Completions API도 충분하지만, 여러 작업을 연결하는 **복잡한 애플리케이션이나 에이전트형 기능**을 만들 때는 Responses API가 더 편리합니다.'

## 정리

이 노트북에서는 03번 Chat Completions 노트북에서 다룬 작업을 Responses API 방식으로 다시 구현했습니다.

- 기본 질문 응답은 `input`에 바로 질문을 넣어 처리했습니다.
- 감정 분석과 PII 추출은 같은 프롬프트 작업을 `responses.create()`로 수행했습니다.
- 출력 형식 고정은 JSON만 출력하도록 지시하고 Python에서 파싱해 검증했습니다.
- 이전 대화 이어가기는 `previous_response_id`를 사용했습니다.

즉, Responses API는 Chat Completions에서 배운 프롬프트 작성 감각을 유지하면서도, 응답 중심의 상태 연결과 확장 흐름을 더 자연스럽게 다룰 수 있게 해줍니다.